In [1]:
import os
import json
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv

load_dotenv()

conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    role=os.getenv("SNOWFLAKE_ROLE"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database="RAW",
)

def run_query(sql):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)

print("Connected!")

Connected!


In [2]:
from IPython.display import HTML

builtin_urls = run_query("""
    SELECT 
        RAW_PAYLOAD:crawl_title::STRING as title,
        RAW_PAYLOAD:source_url::STRING as url
    FROM RAW.BUILTIN.SRC_POSTINGS
    ORDER BY INGESTED_AT DESC
""")

HTML(builtin_urls.to_html(render_links=True, escape=False))

,TITLE,URL
0,Data Engineer - Manager,https://www.builtinnyc.com/job/data-engineer-manager/9657210
1,FP&A Analyst - Data Insights,https://www.builtinnyc.com/job/fp-analyst-data-insights/9662277
2,Data Analyst,https://www.builtinnyc.com/job/data-analyst/9683737
3,"Data Analyst, DOC",https://www.builtinnyc.com/job/data-analyst-doc/9683670
4,Data Analyst/QA Engineer,https://www.builtinnyc.com/job/data-analyst-qa-engineer/9693624
5,Operations Data Analyst- Insurance,https://www.builtinnyc.com/job/operations-data-analyst-insurance/9691977
6,Digital Data Analyst,https://www.builtinnyc.com/job/digital-data-analyst/9688896
7,Marketing and Customer Data Analyst,https://www.builtinnyc.com/job/marketing-and-customer-data-analyst/9688892
8,Market Data Analyst,https://www.builtinnyc.com/job/market-data-analyst/9656987
9,Data Analyst,https://www.builtinnyc.com/job/data-analyst/9640713


In [3]:
# TheirStack seniority coverage
ts_coverage = run_query("""
    SELECT 
        COUNT(*) as total,
        COUNT(CASE WHEN RAW_PAYLOAD:seniority::STRING IS NOT NULL 
                    AND RAW_PAYLOAD:seniority::STRING != '' THEN 1 END) as has_seniority
    FROM RAW.THEIRSTACK.SRC_POSTINGS
""")
ts_coverage["pct_filled"] = (ts_coverage["HAS_SENIORITY"] / ts_coverage["TOTAL"] * 100).round(1)
display(ts_coverage)

# TheirStack seniority distribution
ts_dist = run_query("""
    SELECT 
        RAW_PAYLOAD:seniority::STRING as seniority,
        COUNT(*) as cnt
    FROM RAW.THEIRSTACK.SRC_POSTINGS
    WHERE RAW_PAYLOAD:seniority::STRING IS NOT NULL
      AND RAW_PAYLOAD:seniority::STRING != ''
    GROUP BY 1
    ORDER BY cnt DESC
""")
display(ts_dist)

,TOTAL,HAS_SENIORITY,pct_filled
0,54,54,100.0


,SENIORITY,CNT
0,mid_level,48
1,junior,6


In [4]:
# JSearch entry-level flag distribution
jsearch_flag = run_query("""
    SELECT 
        REGEXP_LIKE(
            RAW_PAYLOAD:job_title::STRING,
            '.*(entry|junior|jr\\.?|associate|new.?grad|early.?career).*',
            'i'
        ) as is_explicitly_entry_level,
        COUNT(*) as cnt
    FROM RAW.JSEARCH.SRC_POSTINGS
    GROUP BY 1
    ORDER BY 1
""")
display(jsearch_flag)

# See the actual titles that would get flagged
jsearch_flagged_titles = run_query("""
    SELECT 
        RAW_PAYLOAD:job_title::STRING as title
    FROM RAW.JSEARCH.SRC_POSTINGS
    WHERE REGEXP_LIKE(
        RAW_PAYLOAD:job_title::STRING,
        '.*(entry|junior|jr\\.?|associate|new.?grad|early.?career).*',
        'i'
    )
    ORDER BY title
""")
display(jsearch_flagged_titles)

,IS_EXPLICITLY_ENTRY_LEVEL,CNT
0,False,322
1,True,13


,TITLE
0,Data Analyst (Entry-Level / Junior)
1,Data Engineer - Financial Crimes - Associate
2,Entry Level Data Engineer
3,Entry Level Data Scientist/Analyst/Java full s...
4,Entry-Level Business Data Analyst
5,Entry-Level Geotechnical Engineer - Field & Da...
6,Entry-Level Infrastructure Operations Engineer
7,"Jr. Business /Data Analyst (Energy, Utility Ex..."
8,Junior Analyst
9,Junior Data Analyst w/ Workday Exp....Rate-$30...


In [5]:
jsearch_seniority_buckets = run_query("""
    SELECT 
        CASE 
            WHEN REGEXP_LIKE(
                RAW_PAYLOAD:job_title::STRING,
                '.*(senior|sr\\.?|lead|principal|staff|manager|director|vp|vice president|avp|head of|architect|chief|svp|evp|gvp|president|officer|executive|leader).*',
                'i'
            ) THEN 'senior_filtered'
            WHEN REGEXP_LIKE(
                RAW_PAYLOAD:job_title::STRING,
                '.*(entry|junior|jr\\.?|new.?grad|early.?career).*',
                'i'
            ) THEN 'explicitly_entry'
            ELSE 'unlabeled_middle'
        END as bucket,
        COUNT(*) as cnt
    FROM RAW.JSEARCH.SRC_POSTINGS
    GROUP BY 1
    ORDER BY cnt DESC
""")
display(jsearch_seniority_buckets)

,BUCKET,CNT
0,unlabeled_middle,175
1,senior_filtered,148
2,explicitly_entry,12
